# Resume Screening Regression Model (SBERT)
Notebook ini membuat model regresi untuk memprediksi `match_score` (1-5) antara CV dan Job Description menggunakan *Sentence-BERT (SBERT)* untuk ekstraksi fitur semantik.

In [1]:
# 1. Import Library
import torch
# Bypass incompatibility bug in transformers 5.x with older PyTorch versions
if not hasattr(torch, 'float8_e8m0fnu'):
    setattr(torch, 'float8_e8m0fnu', torch.float32)

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, ndcg_score
from scipy.stats import spearmanr

import warnings
warnings.filterwarnings('ignore')

d:\Hasil_Coding\NLP_Supervised\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Konfigurasi Penyimpanan SBERT & Cek GPU

In [2]:
# Arahkan Hugging Face cache ke Drive D agar tidak memenuhi Drive C
os.environ["HF_HOME"] = "d:/Hasil_Coding/NLP_Supervised/huggingface_cache"
# Mencegah kernel crash akibat konflik OpenMP (OpenMP Duplicate Runtime)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

gpu_aktif = torch.cuda.is_available()
print("Akselerasi GPU Aktif:", gpu_aktif)
if gpu_aktif:
    print("Nama GPU:", torch.cuda.get_device_name(0))

device = 'cuda' if gpu_aktif else 'cpu'

Akselerasi GPU Aktif: True
Nama GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 2. Load Dataset

In [3]:
file_path = 'NLP_Supervised/resume_job_matching_dataset.csv'
try:
    df = pd.read_csv(file_path)
    print("Dataset berhasil di-load!")
except FileNotFoundError:
    df = pd.read_csv('resume_job_matching_dataset.csv')
    print("Dataset lokal berhasil di-load!")

print("\n--- Shape Dataset ---")
print(df.shape)


Dataset lokal berhasil di-load!

--- Shape Dataset ---
(10000, 3)


## 3. Preprocessing Teks
Untuk SBERT, kita tidak perlu membuang stopwords atau tanda baca secara agresif karena SBERT memahami konteks kalimat utuh. Kita cukup membersihkan karakter aneh dan merapikan spasi.

In [4]:
def clean_text_for_sbert(text):
    if not isinstance(text, str):
        return ""
    # Hapus newline dan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Merapikan teks...")
df['cleaned_resume'] = df['resume'].apply(clean_text_for_sbert)
df['cleaned_jd'] = df['job_description'].apply(clean_text_for_sbert)
print("Selesai.")

Merapikan teks...
Selesai.


## 4. Ekstraksi Fitur (SBERT)
Kita akan mengubah teks menjadi representasi vektor (embedding) 384-dimensi menggunakan `all-MiniLM-L6-v2`.

In [5]:
print("Memuat model SBERT...")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

print("Melakukan encoding Job Description...")
jd_embeddings = model.encode(df['cleaned_jd'].tolist(), show_progress_bar=True, convert_to_numpy=True)

print("Melakukan encoding Resume...")
resume_embeddings = model.encode(df['cleaned_resume'].tolist(), show_progress_bar=True, convert_to_numpy=True)

print("Menghitung Cosine Similarity...")
similarity_scores = np.array([cosine_similarity([resume_embeddings[i]], [jd_embeddings[i]])[0][0] for i in range(len(df))]).reshape(-1, 1)
df['sbert_sim'] = similarity_scores

# Menggabungkan fitur: selisih absolut vektor + skor similaritas
X_diff = np.abs(resume_embeddings - jd_embeddings)
X = np.hstack([X_diff, similarity_scores])
y = df['match_score'].values

print("Dimensi Fitur (X):", X.shape)

Memuat model SBERT...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6745.68it/s]


Melakukan encoding Job Description...


Batches: 100%|██████████| 313/313 [00:05<00:00, 62.53it/s]


Melakukan encoding Resume...


Batches: 100%|██████████| 313/313 [00:04<00:00, 71.03it/s]


Menghitung Cosine Similarity...
Dimensi Fitur (X): (10000, 385)


## 5. Modeling Regresi

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
}

results = []

print("Training dan Evaluasi...")
for name, model_algo in models.items():
    model_algo.fit(X_train, y_train)
    y_pred = model_algo.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    spearman_corr, _ = spearmanr(y_test, y_pred)
    ndcg = ndcg_score([y_test], [y_pred])
    
    results.append({
        'Model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Spearman': spearman_corr,
        'NDCG': ndcg
    })

df_results = pd.DataFrame(results)
display(df_results.sort_values(by='Spearman', ascending=False))

Training dan Evaluasi...


,Model,MAE,RMSE,R2,Spearman,NDCG
0,Linear Regression,0.753037,0.930769,0.358193,0.590794,0.984411
3,LightGBM,0.786630,0.958821,0.318924,0.551590,0.982005
2,XGBoost,0.810125,1.005084,0.251614,0.513131,0.981039
1,Random Forest,0.822005,0.992050,0.270899,0.504554,0.981313


## 6. Hyperparameter Tuning (LGBMRegressor)

In [7]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7]
}

print("Memulai RandomizedSearchCV untuk LightGBM...")
lgbm = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)
random_search = RandomizedSearchCV(lgbm, param_distributions=param_grid, n_iter=10, 
                                   scoring='neg_mean_absolute_error', cv=3, random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

print("Parameter Terbaik:", random_search.best_params_)

best_model = random_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("\n--- Hasil Model Terbaik ---")
print(f"MAE: {mean_absolute_error(y_test, y_pred_best):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_best)):.4f}")
print(f"R2: {r2_score(y_test, y_pred_best):.4f}")
spearman_best, _ = spearmanr(y_test, y_pred_best)
print(f"Spearman Corr: {spearman_best:.4f}")
ndcg_best = ndcg_score([y_test], [y_pred_best])
print(f"NDCG Score: {ndcg_best:.4f}")

Memulai RandomizedSearchCV untuk LightGBM...
Parameter Terbaik: {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.05}

--- Hasil Model Terbaik ---
MAE: 0.7729
RMSE: 0.9466
R2: 0.3362
Spearman Corr: 0.5680
NDCG Score: 0.9842


## 7. Kesimpulan
Penggunaan SBERT memberikan representasi vektor yang kaya akan makna semantik. Anda dapat membandingkan skor evaluasi (Spearman & NDCG) di notebook ini dengan notebook versi TF-IDF untuk melihat seberapa besar peningkatan kualitas *matching*.